# SHAP Analysis for Bias Detection

SHAP analysis for himel7/bias-detector on BABE (sentence-level).

**Outputs:**
- `outputs/shap_samples.jsonl` - per-sentence token attributions
- `outputs/global_word_importance.csv` - global mean|SHAP| per word
- `outputs/examples/shap_*.html` - a few SHAP HTML visualizations


In [3]:
# Imports
import json
import os
import random
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer

import shap


/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utility Functions


In [19]:
# Utility Functions

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def pick_text_column(columns: List[str]) -> str:
    """
    Heuristic selection of the text/sentence column.
    """
    preferred = ["sentence", "text", "content", "sent", "claim"]
    lower = {c.lower(): c for c in columns}
    for p in preferred:
        if p in lower:
            return lower[p]
    # fallback: first string-like column name guess
    for c in columns:
        if any(k in c.lower() for k in ["sent", "text", "content"]):
            return c
    raise ValueError(f"Could not infer text column from: {columns}")


def pick_label_column(columns: List[str]) -> str:
    """
    Heuristic selection of the label column.
    """
    preferred = ["label", "labels", "bias", "is_biased", "sentence_label"]
    lower = {c.lower(): c for c in columns}
    for p in preferred:
        if p in lower:
            return lower[p]
    # fallback: anything that looks like a label
    for c in columns:
        if any(k in c.lower() for k in ["label", "bias"]):
            return c
    raise ValueError(f"Could not infer label column from: {columns}")


def infer_positive_index(id2label: Dict[int, str]) -> int:
    """
    Try to infer which class index corresponds to the "biased" class.
    We search for labels containing 'bias' but not 'no'/'non'.
    Fallback: assume class 1 is positive.
    """
    candidates = []
    for i, lab in id2label.items():
        s = str(lab).lower()
        if "bias" in s and not any(x in s for x in ["no", "non", "unbias", "neutral"]):
            candidates.append(i)
    if len(candidates) == 1:
        return int(candidates[0])
    # If two labels are like "BIASED"/"NOT_BIASED", prefer the one with 'biased'
    for i, lab in id2label.items():
        s = str(lab).lower()
        if "biased" in s and not any(x in s for x in ["not", "no", "non"]):
            return int(i)
    return 1


def softmax_np(logits: np.ndarray) -> np.ndarray:
    x = logits - np.max(logits, axis=1, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=1, keepdims=True)


def normalize_label_values(y: Any) -> np.ndarray:
    """
    Convert labels to {0,1} if possible.
    Handles strings like 'biased', 'no_bias', booleans, and ints.
    """
    if isinstance(y, (list, np.ndarray, pd.Series)):
        y_list = list(y)
    else:
        y_list = [y]

    out = []
    for v in y_list:
        if isinstance(v, (np.integer, int)):
            out.append(int(v))
        elif isinstance(v, (np.floating, float)):
            out.append(int(v))
        elif isinstance(v, (bool, np.bool_)):
            out.append(int(v))
        else:
            s = str(v).strip().lower()
            if s in ["1", "true", "biased", "bias", "yes"]:
                out.append(1)
            elif s in ["0", "false", "not_biased", "no_bias", "unbiased", "neutral", "no"]:
                out.append(0)
            else:
                # last resort: try int conversion
                try:
                    out.append(int(s))
                except Exception as e:
                    raise ValueError(f"Unrecognized label value: {v}") from e

    return np.array(out, dtype=int)


def detokenize_to_words(tokens: List[str]) -> List[str]:
    """
    Simple word reconstruction for RoBERTa/GPT2-style BPE tokens:
    - RoBERTa uses 'Ġ' to mark whitespace before a token.
    - Some tokenizers use '▁' (SentencePiece).
    - We join subword pieces into words based on these markers.
    """
    words = []
    current = ""

    def flush():
        nonlocal current
        if current:
            words.append(current)
            current = ""

    for t in tokens:
        if t in ["[CLS]", "[SEP]"]:
            continue
        # common whitespace markers
        is_new_word = t.startswith("Ġ") or t.startswith("▁")
        clean = t
        clean = clean.replace("Ġ", "")
        clean = clean.replace("▁", "")

        # skip special masking tokens in aggregation
        if clean in ["<s>", "</s>"]:
            continue

        if is_new_word:
            flush()
            current = clean
        else:
            # continuation piece
            if current == "":
                current = clean
            else:
                current += clean

    flush()

    # remove empty or pure punctuation "words" optionally (keep for analysis if you prefer)
    words = [w for w in words if w.strip() != ""]
    return words


# Regex for detecting pure punctuation
_PUNCT_RE = re.compile(r"^\W+$", re.UNICODE)

def aggregate_subword_shap_to_words(tokens: List[str], shap_values: np.ndarray, *, merge_punct_into_previous: bool = True) -> List[Tuple[str, float]]:
    """
    Aggregate token-level SHAP values into word-level attributions using whitespace
    cues in the token strings produced by SHAP's Text masker.

    Works for tokens like "Virginia ", " Democrats", ".", "-" etc.
    Handles both RoBERTa-style markers (Ġ, ▁) and whitespace-based tokens.
    Intelligently detects word boundaries to prevent merging separate words.
    
    tokens: list of token strings (as provided by SHAP masker/tokenizer)
    shap_values: array of per-token SHAP attributions (same length as tokens)
    merge_punct_into_previous: if True, attach punctuation to previous word
    """
    out: List[Tuple[str, float]] = []
    cur_word = ""
    cur_sum = 0.0
    prev_clean = ""  # Track previous token content to detect word boundaries

    def flush():
        nonlocal cur_word, cur_sum, prev_clean
        w = cur_word.strip()
        # Strip trailing punctuation before flushing (we'll reattach if needed)
        if w != "":
            out.append((w, float(cur_sum)))
        cur_word = ""
        cur_sum = 0.0
        prev_clean = ""

    def should_split_word(prev_clean: str, current_clean: str) -> bool:
        """
        Detect if we should split between prev_clean and current_clean.
        Returns True if they should be separate words.
        """
        if not prev_clean or not current_clean:
            return False
        
        # Check for case change indicating word boundary: lowercase->uppercase
        # e.g., "tuesday" followed by "Quoting" -> should split
        if prev_clean and current_clean:
            # If previous token ends with lowercase and current starts with uppercase
            if prev_clean[-1].islower() and current_clean[0].isupper():
                return True
        
        # Check for common word endings in previous token that suggest a complete word
        # followed by what looks like a new word
        common_endings = ['day', 'ing', 'ed', 'ly', 'er', 'tion', 'ment', 'ism', 'ist', 'ous', 'ful', 'ness']
        prev_lower = prev_clean.lower()
        for ending in common_endings:
            if prev_lower.endswith(ending) and len(prev_clean) >= len(ending) + 2:
                # Previous looks like a complete word, and current starts with lowercase letter
                if current_clean[0].isalpha() and current_clean[0].islower():
                    # Check if current token looks like it could start a new word (has vowels)
                    if any(c in current_clean.lower()[:3] for c in 'aeiou'):
                        return True
        
        # Check for punctuation in current token that indicates separation
        # If current token starts with punctuation that's not a hyphen (hyphens can be in words)
        if current_clean and current_clean[0] in ',.!?;:()[]{}"\'`':
            return True
            
        return False

    for t, s in zip(tokens, shap_values):
        if t is None:
            continue

        # SHAP often includes boundary "" tokens
        if t == "":
            flush()
            continue

        # Preserve the raw token for whitespace checks
        raw = t
        
        # Handle RoBERTa-style markers (Ġ, ▁) - strip them for content but check for word boundary
        has_roberta_marker = raw.startswith("Ġ") or raw.startswith("▁")
        if has_roberta_marker:
            # This starts a new word
            flush()
            clean = raw.replace("Ġ", "").replace("▁", "")
            prev_clean = ""  # Reset since we flushed
        else:
            # Strip whitespace but preserve content
            clean = raw.strip()
            # Check if we should split based on word boundary detection
            if prev_clean and should_split_word(prev_clean, clean):
                flush()
                prev_clean = ""

        # Skip special tokens if they appear
        if clean in {"[CLS]", "[SEP]", "<s>", "</s>"}:
            flush()
            continue

        # If token begins with whitespace (and no RoBERTa marker), it likely starts a new word
        starts_new_word = (not has_roberta_marker and len(raw) > 0 and raw[0].isspace())

        # Handle punctuation tokens
        is_punct = bool(_PUNCT_RE.match(clean)) and clean != ""
        
        # Handle hyphens specially - they can be word separators or part of words
        is_hyphen = clean in ['-', '—', '–', '‑']

        if starts_new_word:
            flush()
            prev_clean = ""

        if is_punct and merge_punct_into_previous and out and not is_hyphen:
            # Attach punctuation to previous word in output (more readable)
            # But hyphens might indicate compound words that should be split
            prev_w, prev_s = out[-1]
            out[-1] = (prev_w + clean, float(prev_s + s))
            prev_clean = clean  # Update tracking
            continue
        elif is_hyphen:
            # Hyphen might be a word separator - flush current word
            # The next token will start a new word
            if cur_word:
                flush()
            prev_clean = ""
            continue

        # Add token to current word buffer
        cur_word += clean
        cur_sum += float(s)
        prev_clean = clean  # Track for next iteration

        # If token ends with whitespace (and no RoBERTa marker), that word is complete
        if not has_roberta_marker and len(raw) > 0 and raw[-1].isspace():
            flush()
            prev_clean = ""

    flush()
    return out


## Configuration

Modify these parameters as needed:


In [23]:
# Configuration parameters
# Model selection: Choose one model to analyze
# Option 1: Original model
model_name = "mediabiasgroup/da-roberta-babe-ft" #"himel7/bias-detector"
model_suffix = "daroberta"  # Empty suffix for original model

# Option 2: DA-RoBERTa model (uncomment to use)
# model_name = "mediabiasgroup/da-roberta-babe-ft"
# model_suffix = "daroberta"  # Suffix to distinguish outputs

dataset_name = "mediabiasgroup/BABE"
split = "test"
text_col = ""  # Leave empty for auto-detection
label_col = ""  # Leave empty for auto-detection
max_length = 256

# Sampling parameters
n_tp = 100  # Number of True Positives to sample
n_fp = 100  # Number of False Positives to sample
n_tn = 100  # Number of True Negatives to sample

# SHAP parameters
# NOTE: SHAP is computationally expensive. Options:
# - 30: Quick analysis (current)
# - 100-150: More comprehensive, better for paper (recommended)
# - 300: Full stratified sample (very slow, may take hours)
max_shap_examples = 300  # Increased for more comprehensive analysis
seed = 7
batch_infer = 64  # Batch size for inference

# Output directory - will add suffix to output files
output_dir = "outputs"
# Add suffix to output filenames if specified
if model_suffix:
    output_suffix = f"_{model_suffix}"
else:
    output_suffix = ""


## Setup


In [7]:
# Setup: seed, directories, device
set_seed(seed)
ensure_dir(output_dir)
ensure_dir(os.path.join(output_dir, "examples"))

device = torch.device("cuda" if torch.cuda.is_available() else "mps")
print(f"Using device: {device}")


Using device: mps


## Load Model and Tokenizer


In [24]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()
model.to(device)

id2label = model.config.id2label if hasattr(model.config, "id2label") else {0: "NEG", 1: "POS"}
positive_index = infer_positive_index(id2label)
print(f"[INFO] id2label={id2label} | positive_index={positive_index} ({id2label.get(positive_index)})")


[INFO] id2label={0: 'LABEL_0', 1: 'LABEL_1'} | positive_index=1 (LABEL_1)


## Load Dataset


In [9]:
# Load dataset split
ds = load_dataset(dataset_name, split=split)
cols = ds.column_names
text_col_actual = text_col.strip() or pick_text_column(cols)
label_col_actual = label_col.strip() or pick_label_column(cols)
print(f"[INFO] Using text_col='{text_col_actual}' label_col='{label_col_actual}' from columns={cols}")

texts = [str(x) for x in ds[text_col_actual]]
y_true = normalize_label_values(ds[label_col_actual])
print(f"[INFO] Loaded {len(texts)} samples")


[INFO] Using text_col='text' label_col='label' from columns=['text', 'outlet', 'label', 'topic', 'news_link', 'biased_words', 'uuid', 'type', 'label_opinion']
[INFO] Loaded 1000 samples


## Run Inference


In [25]:
# Batch inference function
def run_inference(model, tokenizer, texts: List[str], device: torch.device, max_length: int) -> np.ndarray:
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits.detach().cpu().numpy()
    probs = softmax_np(logits)
    return probs

# Run inference on all texts
probs_all = []
for start in range(0, len(texts), batch_infer):
    batch = texts[start:start + batch_infer]
    probs = run_inference(model, tokenizer, batch, device, max_length)
    probs_all.append(probs)
    if (start // batch_infer + 1) % 10 == 0:
        print(f"Processed {start + len(batch)}/{len(texts)} samples...")
probs_all = np.vstack(probs_all)

p_pos = probs_all[:, positive_index]
y_pred = (p_pos >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
print(f"[INFO] Confusion matrix @0.5: TP={tp} FP={fp} TN={tn} FN={fn}")


Processed 640/1000 samples...
[INFO] Confusion matrix @0.5: TP=394 FP=12 TN=429 FN=165


## Sample Instances for SHAP


In [26]:
# Data class for sampled items
@dataclass
class SampledItem:
    text: str
    gold: int
    pred: int
    prob_pos: float

# Stratified sampling function
def stratified_sample(texts: List[str], y_true: np.ndarray, y_pred: np.ndarray, p_pos: np.ndarray,
                      n_tp: int, n_fp: int, n_tn: int, seed: int) -> List[SampledItem]:
    """
    Sample TP / FP / TN instances for SHAP.
    """
    rng = np.random.default_rng(seed)
    tp_idx = np.where((y_true == 1) & (y_pred == 1))[0]
    fp_idx = np.where((y_true == 0) & (y_pred == 1))[0]
    tn_idx = np.where((y_true == 0) & (y_pred == 0))[0]

    def pick(idx: np.ndarray, n: int) -> np.ndarray:
        if len(idx) == 0:
            return np.array([], dtype=int)
        n_eff = min(n, len(idx))
        return rng.choice(idx, size=n_eff, replace=False)

    chosen = np.concatenate([pick(tp_idx, n_tp), pick(fp_idx, n_fp), pick(tn_idx, n_tn)])
    rng.shuffle(chosen)

    items: List[SampledItem] = []
    for i in chosen:
        items.append(SampledItem(
            text=texts[i],
            gold=int(y_true[i]),
            pred=int(y_pred[i]),
            prob_pos=float(p_pos[i]),
        ))
    return items

# Sample instances
sampled = stratified_sample(
    texts=texts,
    y_true=y_true,
    y_pred=y_pred,
    p_pos=p_pos,
    n_tp=n_tp,
    n_fp=n_fp,
    n_tn=n_tn,
    seed=seed
)
if len(sampled) == 0:
    raise RuntimeError("No samples selected. Check label normalization and thresholding.")
print(f"[INFO] Selected {len(sampled)} samples (requested TP/FP/TN = {n_tp}/{n_fp}/{n_tn}).")

# Show distribution before limiting
from collections import Counter
pred_types_before = Counter([
    'TP' if s.gold == 1 and s.pred == 1 else
    'FP' if s.gold == 0 and s.pred == 1 else
    'TN' if s.gold == 0 and s.pred == 0 else 'FN'
    for s in sampled
])
print(f"[INFO] Distribution before limiting: {dict(pred_types_before)}")

# Cost control: SHAP is expensive
sampled = sampled[: min(len(sampled), max_shap_examples)]
print(f"[INFO] Running SHAP on {len(sampled)} samples (max_shap_examples={max_shap_examples}).")

# Show final distribution
pred_types_after = Counter([
    'TP' if s.gold == 1 and s.pred == 1 else
    'FP' if s.gold == 0 and s.pred == 1 else
    'TN' if s.gold == 0 and s.pred == 0 else 'FN'
    for s in sampled
])
print(f"[INFO] Final distribution: {dict(pred_types_after)}")
print(f"[INFO] Note: With max_shap_examples={max_shap_examples}, you'll get comprehensive coverage, especially of FPs.")


[INFO] Selected 212 samples (requested TP/FP/TN = 100/100/100).
[INFO] Distribution before limiting: {'TP': 100, 'TN': 100, 'FP': 12}
[INFO] Running SHAP on 212 samples (max_shap_examples=300).
[INFO] Final distribution: {'TP': 100, 'TN': 100, 'FP': 12}
[INFO] Note: With max_shap_examples=300, you'll get comprehensive coverage, especially of FPs.


## Setup SHAP Explainer


In [27]:
# Build prediction function for SHAP
def build_predict_fn(model, tokenizer, device: torch.device, max_length: int, positive_index: int):
    """
    Returns a function f(texts: List[str]) -> np.ndarray of shape (n, 2)
    containing probabilities for each class, suitable for SHAP multi-output explanation.
    """
    def predict(texts) -> np.ndarray:
        # Handle different input formats
        if isinstance(texts, str):
            texts = [texts]
        elif not isinstance(texts, list):
            texts = list(texts)
        
        # Ensure all elements are strings
        texts = [str(t) if not isinstance(t, str) else t for t in texts]
        
        enc = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits.detach().cpu().numpy()
        probs = softmax_np(logits)
        return probs
    return predict

# Create SHAP explainer with Text masker
# The Text masker handles tokenization and masking for transformer models
try:
    # Try creating masker with tokenizer
    masker = shap.maskers.Text(tokenizer)
except Exception as e:
    print(f"Warning: Could not create Text masker with tokenizer: {e}")
    # Fallback: create masker without tokenizer (less efficient)
    masker = shap.maskers.Text()

predict_fn = build_predict_fn(
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_length=max_length,
    positive_index=positive_index
)

# Create explainer - try with output_names first
try:
    explainer = shap.Explainer(predict_fn, masker, output_names=list(id2label.values()))
except Exception as e:
    print(f"Warning: Could not create explainer with output_names: {e}")
    # Fallback: create without output_names
    explainer = shap.Explainer(predict_fn, masker)

print("[INFO] SHAP explainer created")


[INFO] SHAP explainer created


## Run SHAP Analysis

**Note:** This may take a while depending on the number of samples and model complexity.


In [28]:
# Run SHAP on sampled texts
sample_texts = [str(s.text) for s in sampled]  # Ensure all are strings
print(f"[INFO] Running SHAP on {len(sample_texts)} samples...")

# Try to run SHAP on all texts at once first (most efficient)
# If that fails, fall back to processing one at a time
try:
    print("Attempting to run SHAP on all samples at once...")
    explanation = explainer(sample_texts)
    print("[INFO] SHAP analysis complete (batch mode)")
except (ValueError, TypeError) as e:
    print(f"Batch mode failed: {e}")
    print("Falling back to processing samples one at a time...")
    
    # Process texts one at a time and combine results
    all_values_list = []
    all_data_list = []
    
    for i, text in enumerate(sample_texts):
        if (i + 1) % 5 == 0:
            print(f"  Processing sample {i + 1}/{len(sample_texts)}...")
        
        # Ensure text is a clean string
        text_clean = str(text).strip()
        if not text_clean:
            print(f"  Warning: Sample {i+1} is empty, skipping...")
            continue
        
        # Try different input formats
        exp = None
        for attempt_format in [[text_clean], text_clean]:
            try:
                exp = explainer(attempt_format)
                break
            except (ValueError, TypeError):
                continue
        
        if exp is None:
            raise RuntimeError(f"Could not process sample {i+1} with any input format")
        
        # Extract values - handle different shapes
        # Result should have shape (1, n_tokens, n_outputs) for a single text
        if exp.values.ndim == 3:
            # Shape (1, n_tokens, n_outputs) -> take [0] to get (n_tokens, n_outputs)
            vals = exp.values[0]
        elif exp.values.ndim == 2:
            # Shape (n_tokens, n_outputs) - use directly
            vals = exp.values
        else:
            raise RuntimeError(f"Unexpected SHAP values shape: {exp.values.shape}")
        all_values_list.append(vals)
        
        # Extract data (tokens) - should be a list/array
        if hasattr(exp, 'data'):
            # exp.data should be a list with one element (the tokens for this text)
            if isinstance(exp.data, (list, tuple)):
                if len(exp.data) > 0:
                    # Take the first element which contains the tokens
                    data_item = exp.data[0]
                else:
                    data_item = exp.data
            elif isinstance(exp.data, np.ndarray):
                # If it's an array, take first element if 2D or 3D
                if exp.data.ndim == 3:
                    data_item = exp.data[0]  # (1, n_tokens) -> (n_tokens,)
                elif exp.data.ndim == 2:
                    data_item = exp.data[0] if exp.data.shape[0] == 1 else exp.data
                else:
                    data_item = exp.data
            else:
                data_item = exp.data
        else:
            # Fallback: try to get tokens from the explanation
            data_item = None
        all_data_list.append(data_item)
    
    # Stack values to create (n_samples, n_tokens, n_outputs) shape
    all_values = np.stack(all_values_list)
    
    # Create a combined explanation object that matches the expected structure
    explanation = shap.Explanation(
        values=all_values,
        data=all_data_list
    )
    
    print("[INFO] SHAP analysis complete (sequential mode)")


[INFO] Running SHAP on 212 samples...
Attempting to run SHAP on all samples at once...


PartitionExplainer explainer: 213it [26:44,  7.61s/it]                         


[INFO] SHAP analysis complete (batch mode)


## Export Results


In [29]:
# Export per-sentence attributions + compute global word importance
jsonl_path = os.path.join(output_dir, f"shap_samples{output_suffix}.jsonl")
global_rows = []

with open(jsonl_path, "w", encoding="utf-8") as f:
    for i, s in enumerate(sampled):
        # tokens for this sample
        tokens = list(explanation.data[i])
        # shap values for the positive class
        # explanation.values shape can be (n, tokens, outputs)
        vals = explanation.values[i]
        if vals.ndim == 2:
            # (tokens, outputs)
            shap_pos = vals[:, positive_index]
        else:
            raise RuntimeError(f"Unexpected SHAP values shape per-sample: {vals.shape}")

        # word aggregation
        word_attribs = aggregate_subword_shap_to_words(tokens, shap_pos)

        record = {
            "text": s.text,
            "gold": s.gold,
            "pred": s.pred,
            "prob_pos": s.prob_pos,
            "tokens": tokens,
            "shap_pos": [float(x) for x in shap_pos.tolist()],
            "word_attribs": [{"word": w, "shap": v} for (w, v) in word_attribs],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

        for (w, v) in word_attribs:
            # normalize the "word" a bit for global stats
            w_norm = re.sub(r"\s+", "", w.strip().lower())
            if w_norm == "":
                continue
            global_rows.append({"word": w_norm, "abs_shap": abs(float(v)), "shap": float(v)})

print(f"[INFO] Wrote per-sentence SHAP to: {jsonl_path}")


[INFO] Wrote per-sentence SHAP to: outputs/shap_samples_daroberta.jsonl


## Compute Global Word Importance


In [30]:
# Compute global word importance
gdf = pd.DataFrame(global_rows)
if len(gdf) == 0:
    raise RuntimeError("No global rows collected; check tokenizer markers and SHAP output.")

global_imp = (
    gdf.groupby("word")
    .agg(mean_abs_shap=("abs_shap", "mean"),
         mean_shap=("shap", "mean"),
         count=("word", "size"))
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index()
)
csv_path = os.path.join(output_dir, f"global_word_importance{output_suffix}.csv")
global_imp.to_csv(csv_path, index=False)
print(f"[INFO] Wrote global word importance to: {csv_path}")

# Display top words
print("\nTop 20 words by mean absolute SHAP value:")
print(global_imp.head(20))


[INFO] Wrote global word importance to: outputs/global_word_importance_daroberta.csv

Top 20 words by mean absolute SHAP value:
             word  mean_abs_shap  mean_shap  count
0   heartlessness       0.767285   0.767285      1
1      flippantly       0.634442   0.634442      1
2          lashed       0.483722   0.483722      1
3    antisemitic.       0.479918   0.479918      1
4         dubious       0.465581   0.465581      1
5       flippancy       0.452473   0.452473      1
6           nasty       0.405461   0.405461      1
7         blasted       0.383722   0.383722      1
8         junkies       0.376823   0.376823      1
9         sandbag       0.368089   0.368089      1
10       guzzling       0.363833   0.363833      1
11  predominantly       0.338808   0.338808      1
12       gruesome       0.338780   0.338780      1
13           mobs       0.337247   0.337247      1
14        throes,       0.329295   0.329295      1
15    astonishing       0.328147   0.328147      1
16   

## Generate HTML Visualizations


In [31]:
# Save a few HTML visualizations
# Note: shap.plots.text() returns HTML content directly as a string
for i in range(min(5, len(sampled))):
    html_path = os.path.join(output_dir, "examples", f"shap{output_suffix}_{i}.html")
    # Create an Explanation object for this sample and positive class
    # Get values for this sample and positive class: shape (n_tokens,)
    shap_values_i = explanation.values[i][:, positive_index]
    # Get tokens for this sample
    tokens_i = explanation.data[i]
    # Create a single-output Explanation object for the text plot
    exp_i = shap.Explanation(
        values=shap_values_i,
        data=tokens_i,
        base_values=explanation.base_values[i][positive_index] if hasattr(explanation, 'base_values') and explanation.base_values is not None else None
    )
    # shap.plots.text() with display=False returns HTML as a string
    html_content = shap.plots.text(exp_i, display=False)
    # Write the HTML content directly to file
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html_content)
    print(f"[INFO] Wrote example HTML: {html_path}")

print("\n[DONE] Analysis complete!")


[INFO] Wrote example HTML: outputs/examples/shap_daroberta_0.html
[INFO] Wrote example HTML: outputs/examples/shap_daroberta_1.html
[INFO] Wrote example HTML: outputs/examples/shap_daroberta_2.html
[INFO] Wrote example HTML: outputs/examples/shap_daroberta_3.html
[INFO] Wrote example HTML: outputs/examples/shap_daroberta_4.html

[DONE] Analysis complete!
